# Lesson 8 — Persistent Memory & Checkpointers

## What you will learn
- Why graphs are **stateless by default** and how to fix that
- `MemorySaver` — in-memory (lost on restart, great for testing)
- `SqliteSaver` — disk-based (survives restarts, production-like)
- `thread_id` — how to separate users/sessions
- How to **inspect and replay** state history
- **User profile memory** — storing facts across conversations

## Core concept
```
WITHOUT checkpointer:  invoke() → runs → forgets everything

WITH checkpointer:     invoke() → runs → saves snapshot
                       invoke() → loads snapshot → continues

thread_id = "user-123"  →  all checkpoints for this user's session
thread_id = "user-456"  →  completely isolated session
```

In [ ]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver

llm = ChatOllama(model="llama3.2", temperature=0.7)


class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot_node(state: ChatState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}


def build_graph(checkpointer=None):
    b = StateGraph(ChatState)
    b.add_node("chatbot", chatbot_node)
    b.add_edge(START, "chatbot")
    b.add_edge("chatbot", END)
    return b.compile(checkpointer=checkpointer)


print("Setup complete!")

---
## Part 1 — Without Checkpointer (Stateless)

Notice how the AI forgets everything between calls.

In [ ]:
graph_stateless = build_graph()  # no checkpointer

r1 = graph_stateless.invoke({"messages": [HumanMessage(content="My name is Ahmed.")]})
print(f"Turn 1: {r1['messages'][-1].content[:100]}")

r2 = graph_stateless.invoke({"messages": [HumanMessage(content="What is my name?")]})
print(f"Turn 2: {r2['messages'][-1].content[:100]}")
print("⚠️  No memory — it forgot!")

---
## Part 2 — MemorySaver (In-Session Memory)

Same `thread_id` = same conversation = AI remembers.

In [ ]:
graph_mem = build_graph(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "ahmed-session"}}

# Turn 1
r1 = graph_mem.invoke({"messages": [HumanMessage(content="My name is Ahmed. I love Python.")]}, config=cfg)
print(f"Turn 1: {r1['messages'][-1].content[:120]}")

# Turn 2 — AI automatically loads previous state
r2 = graph_mem.invoke({"messages": [HumanMessage(content="What is my name and what do I love?")]}, config=cfg)
print(f"Turn 2: {r2['messages'][-1].content[:120]}")
print("✅ It remembered!")

In [ ]:
# Different thread_id = different user = no memory of Ahmed
cfg_bob = {"configurable": {"thread_id": "bob-session"}}
r3 = graph_mem.invoke({"messages": [HumanMessage(content="What is my name?")]}, config=cfg_bob)
print(f"Different user: {r3['messages'][-1].content[:100]}")
print("✅ Isolated — Bob knows nothing about Ahmed")

---
## Part 3 — SqliteSaver (Disk Persistence)

Survives Python restarts. Run this cell, restart kernel, run again.

In [ ]:
DB_FILE = "session_checkpoints.db"

with SqliteSaver.from_conn_string(DB_FILE) as ckpt:
    graph_sql = build_graph(checkpointer=ckpt)
    cfg_sql = {"configurable": {"thread_id": "persistent-user-1"}}

    # Check history
    history = list(graph_sql.get_state_history(cfg_sql))
    if history:
        print(f"Found {len(history)} existing checkpoints — resuming!")
    else:
        print("New session — saving first turn...")
        graph_sql.invoke(
            {"messages": [HumanMessage(content="Remember: I am a data engineer who loves LangGraph.")]},
            config=cfg_sql
        )

    result = graph_sql.invoke(
        {"messages": [HumanMessage(content="What do you know about me?")]},
        config=cfg_sql
    )
    print(f"AI: {result['messages'][-1].content[:200]}")
    print(f"\n✅ Saved to {DB_FILE} — restart kernel and re-run to verify persistence!")

---
## Part 4 — Inspect State History

In [ ]:
graph_hist = build_graph(checkpointer=MemorySaver())
cfg_hist   = {"configurable": {"thread_id": "history-demo"}}

for msg_text in ["I am Sara.", "I work in AI.", "What do you know about me?"]:
    graph_hist.invoke({"messages": [HumanMessage(content=msg_text)]}, config=cfg_hist)

# Current state
state = graph_hist.get_state(cfg_hist)
print(f"Current conversation ({len(state.values['messages'])} messages):")
for m in state.values["messages"]:
    role = "Sara" if isinstance(m, HumanMessage) else "AI"
    print(f"  [{role}]: {m.content[:80]}")

In [ ]:
# All checkpoints (time-travel debugging!)
print("Checkpoint history:")
for i, snap in enumerate(graph_hist.get_state_history(cfg_hist)):
    n = len(snap.values.get("messages", []))
    print(f"  [{i}] {n} messages | next node: {snap.next}")

---
## Part 5 — User Profile Memory Pattern

Store facts about the user in a dedicated `user_profile` state field.  
This field grows across turns and personalizes responses.

In [ ]:
class ProfileState(TypedDict):
    messages:     Annotated[list, add_messages]
    user_profile: dict


def profile_node(state: ProfileState) -> dict:
    profile = state.get("user_profile", {})
    profile_text = "\n".join(f"  {k}: {v}" for k, v in profile.items()) or "  (none yet)"
    system = SystemMessage(content=f"Known about user:\n{profile_text}\nUse this to personalize responses.")
    response = llm.invoke([system] + state["messages"])

    # Simple fact extraction
    text = state["messages"][-1].content.lower()
    new_profile = dict(profile)
    if "my name is" in text:
        new_profile["name"] = text.split("my name is")[1].split(".")[0].strip().title()
    if "i love" in text:
        new_profile["loves"] = text.split("i love")[1].split(".")[0].strip()
    if "i work" in text:
        new_profile["job"] = text.split("i work")[1].split(".")[0].strip()

    return {"messages": [response], "user_profile": new_profile}


b_p = StateGraph(ProfileState)
b_p.add_node("chatbot", profile_node)
b_p.add_edge(START, "chatbot")
b_p.add_edge("chatbot", END)
graph_p = b_p.compile(checkpointer=MemorySaver())
cfg_p   = {"configurable": {"thread_id": "profile-demo"}}

for turn in ["My name is Ahmed. I love machine learning.", "I work as a data engineer.", "What do you know about me?"]:
    print(f"\nYou: {turn}")
    r = graph_p.invoke({"messages": [HumanMessage(content=turn)], "user_profile": {}}, config=cfg_p)
    s = graph_p.get_state(cfg_p)
    print(f"AI: {r['messages'][-1].content[:150]}")
    print(f"Profile: {s.values.get('user_profile', {})}")

## Key Takeaways

| Checkpointer | Storage | Survives restart? | Use for |
|-------------|---------|-------------------|---------|
| `None` | None | No | Stateless single-turn tasks |
| `MemorySaver()` | RAM | No | Development, testing, demos |
| `SqliteSaver` | Disk (`.db`) | ✅ Yes | Production single-server apps |

| Concept | Description |
|---------|-------------|
| `thread_id` | Each value = separate isolated session |
| `graph.get_state(config)` | Current state snapshot |
| `graph.get_state_history(config)` | All historical snapshots |
| `user_profile` field | Grow facts across turns without replacing messages |

## 🏋️ Exercise
1. Build a **personal notebook** assistant:
   - State has `messages` + `notes: list`
   - When user says "add note: X" → append to `notes`
   - When user says "show notes" → list all notes from state
   - Use `SqliteSaver` so notes survive restarts
2. Test by adding 3 notes, restarting kernel, then running "show my notes"

In [ ]:
# Your notebook assistant here
